# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [35]:
# imports
from IPython.display import display
from openai import OpenAI
import gradio as gr
import json

# constants
MODEL = 'llama3.2'

# set up environment
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

system_prompt =  """
You get asked a technical question, and needs to respond with detailed information.
"""

price_function = {
    "name": "calculator",
    "description": "Use this tool to solve simple math expressions.",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The simple simple math expression to solve.",
            },
        },
        "required": ["expression"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]

def calculator(expression):
    f"Tool calculator called with {expression}"
    try:
        # Only allow numbers and basic operators
        allowed_chars = "0123456789+-*/(). "
        if any(c not in allowed_chars for c in expression):
            return "Error: Invalid characters"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            expression = arguments.get('expression')
            solution = calculator(expression)
            responses.append({
                "role": "tool",
                "content": f"The math expression solutions is {solution}",
                "tool_call_id": tool_call.id
            })
    return responses

def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history
    response = ollama.chat.completions.create(model=MODEL, messages=messages, stream=True)
         
    # while response.choices[0].finish_reason=="tool_calls":
    #     message = response.choices[0].message
    #     responses = handle_tool_calls(message)
    #     messages.append(message)
    #     messages.extend(responses)
    #     response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # reply = response.choices[0].message.content
    # history += [{"role":"assistant", "content":reply}]
    # return history
    
    partial = ""
    history.append({"role": "assistant", "content": ""})
    for chunk in response:
        delta = chunk.choices[0].delta.content or ""
        partial += delta
        history[-1]["content"] = partial
        yield history


def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

def change_model(model):
    MODEL = model

# UI definition
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=400, type="messages")
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")
        model = gr.Dropdown(choices=[("Ollama", "llama3.2"), ("Gemini", "gemini-2.5-flash")], value="llama3.2", label="Select model")
        model.change(fn=change_model, inputs=model)

# Hooking up events to callbacks
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7920
* To create a public link, set `share=True` in `launch()`.
